# 🟠 MODELO 3: LSTM + Dropout (Regularización)

## 📋 Objetivo del ejercicio
Prevenir **overfitting** añadiendo Dropout al modelo LSTM.

## ⚠️ Problema del Modelo 2
El LSTM puede **memorizar** los datos de entrenamiento y no generalizar bien.

## ✅ Solución: Dropout
- **Dropout** apaga aleatoriamente neuronas durante el entrenamiento
- Fuerza al modelo a aprender representaciones **robustas**
- Reduce dependencia de neuronas específicas

### 📚 Teoría de examen:
> "Dropout es una técnica de regularización que previene overfitting.
> Se usa con tasas entre 0.2-0.5 (20%-50% de neuronas desactivadas).
> Solo se aplica durante ENTRENAMIENTO, no en inferencia."

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

seed = 42
np.random.seed(seed)

In [ ]:
train = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/train.csv", index_col="row_id")
test = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/test.csv", index_col="row_id")

print(f"📊 Datos cargados")

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

tf.random.set_seed(seed)

X = train['message'].values
y = train['spam_label'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

MAX_WORDS = 5000
MAX_LEN = 100
EMBEDDING_DIM = 64

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN, padding='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(test['message'].values), maxlen=MAX_LEN, padding='post')

print("✅ Preprocesamiento completo")

## 🏗️ Modelo LSTM + Dropout

### 📚 ¿Dónde colocar Dropout?
1. **Después del Embedding**: Previene overfitting en embeddings
2. **Después del LSTM**: Regulariza la salida de la capa recurrente
3. **Antes de la capa Dense final**: Regulariza la clasificación

### 🔑 Pregunta de examen típica:
**P: ¿Qué tasa de Dropout usar?**

**R: Típicamente:**
- **0.2-0.3** después de Embedding (ligero)
- **0.3-0.5** después de LSTM (moderado)
- **0.2-0.3** antes de Dense (ligero)

**Si hay mucho overfitting → aumentar. Si underfitting → reducir.**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    Dropout(0.3),  # ✨ MEJORA 1: Dropout después del embedding
    
    LSTM(64),
    Dropout(0.4),  # ✨ MEJORA 2: Dropout después del LSTM (tasa mayor)
    
    Dense(16, activation='relu'),
    Dropout(0.3),  # ✨ MEJORA 3: Dropout antes de la salida
    
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

### 📊 Observación importante:
Con Dropout, el modelo tiene los **mismos parámetros** que sin Dropout.
Dropout NO añade parámetros, solo modifica el entrenamiento.

In [ ]:
# Entrenar con MÁS épocas porque Dropout hace el entrenamiento más lento
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,  # ⬆️ Más épocas
    batch_size=32,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# Visualizar overfitting (o falta de él)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy: Train vs Validation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss: Train vs Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📌 Si las curvas están cerca → Dropout funciona (menos overfitting)")
print("📌 Si train >> val → Aún hay overfitting (aumentar Dropout)")
print("📌 Si train ≈ val pero ambas bajas → Underfitting (reducir Dropout)")

In [ ]:
val_loss, val_acc = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 Validation Accuracy: {val_acc:.4f}")
print(f"📊 Validation Loss: {val_loss:.4f}")

In [ ]:
y_pred_proba = model.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

submission = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/sample_submission.csv")
submission["spam_label"] = y_pred
submission.to_csv('submission_lstm_dropout.csv', index=False)

print("✅ Submission creado: submission_lstm_dropout.csv")

---

## 📚 Resumen para Examen

### ✅ Ventajas Dropout:
- **Previene overfitting** efectivamente
- **No añade parámetros** al modelo
- Fuerza **robustez** y **generalización**
- Fácil de implementar

### ❌ Desventaja:
- **Entrenamiento más lento** (necesita más épocas)
- Puede causar **underfitting** si la tasa es muy alta

### 🎯 Cuándo usar Dropout:
- Val accuracy << Train accuracy (overfitting claro)
- Dataset pequeño/mediano
- Modelo con muchos parámetros

### 🔄 Siguiente paso:
**Usar Bidirectional LSTM** para capturar contexto en ambas direcciones.